In [1]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set the project root directory as the current directory for easier module imports
project_root = os.path.abspath("..")
sys.path.append(project_root)


In [4]:
import json
from pathlib import Path
import pandas as pd

def lm_eval_results_to_df(src) -> pd.DataFrame:
    """
    将 lm-eval-harness 的原始 JSON 结果转换为易读的 pandas DataFrame。

    参数
    ----
    src : str | Path | dict
        - 若为 str / Path：表示 JSON 文件路径；
        - 若为 dict：表示已加载的 JSON 对象。

    返回
    ----
    pd.DataFrame
        列包含：Task, Version, Filter, Shot, Metric, Arrow, Value, Stderr
    """
    # ---------- 1. 读入 JSON ----------
    if isinstance(src, (str, Path)):
        data = json.loads(Path(src).read_text())
    elif isinstance(src, dict):
        data = src
    else:
        raise TypeError("src 必须是文件路径或已解析的 dict")

    # ---------- 2. 解析结果 ----------
    rows = []
    task_order = {t: i for i, t in enumerate(data["results"].keys())}
    for task, res in data["results"].items():
        ver  = data.get("versions", {}).get(task, "")
        shot = data.get("n-shot", {}).get(task, "")
        hib  = data.get("higher_is_better", {}).get(task, {})
        for key, val in res.items():
            if "_stderr" in key or "," not in key:
                continue
            metric, flt = key.split(",", 1)
            stderr_val  = res.get(f"{metric}_stderr,{flt}", "")
            rows.append(
                dict(
                    Task=task,
                    Version=ver,
                    Filter=flt,
                    Shot=shot,
                    Metric=metric,
                    Arrow="↑" if hib.get(metric, True) else "↓",
                    Value=val,
                    Stderr=stderr_val,
                    _order=task_order[task],
                )
            )

    # ---------- 3. 生成 DataFrame ----------
    df = (
        pd.DataFrame(rows)
          .sort_values(["_order", "Metric"])
          .drop(columns="_order")
          .reset_index(drop=True)
    )
    return df


In [6]:
# 直接读取文件
df = lm_eval_results_to_df(
    "/mnt/public/code/hanyu/codes/SEAP/eval_out/__mnt__public__model__huggingface__Llama-2-7b-hf/results_2025-05-01T15-36-59.055314.json"
)
df

,Task,Version,Filter,Shot,Metric,Arrow,Value,Stderr
0,arc_challenge,1.0,none,0,acc,↑,0.434300,0.014485
1,arc_challenge,1.0,none,0,acc_norm,↑,0.462457,0.014570
2,arc_easy,1.0,none,0,acc,↑,0.763047,0.008725
3,arc_easy,1.0,none,0,acc_norm,↑,0.745370,0.008939
4,boolq,2.0,none,0,acc,↑,0.777370,0.007276
5,gsm8k,3.0,strict-match,5,exact_match,↑,0.137225,0.009478
6,gsm8k,3.0,flexible-extract,5,exact_match,↑,0.141016,0.009587
7,hellaswag,1.0,none,0,acc,↑,0.571699,0.004938
8,hellaswag,1.0,none,0,acc_norm,↑,0.760307,0.004260
9,mathqa,1.0,none,0,acc,↑,0.280737,0.008226
